# EchoGesture 01: landmark extraction (run with Save Version, then Save & Run All)

**Session options:** Accelerator **None** (MediaPipe runs on CPU; this saves GPU quota), Internet **ON**.  
**Inputs:** `echogesture` (newest version), `risangbaskoro/wlasl-processed`, competition `asl-signs`.

Run it with **Save Version, then Save & Run All (Commit)** so it keeps running with the browser closed. Everything written to `/kaggle/working/data` becomes this notebook's **output**, which the training notebook (02) adds as an input. Expect about 2-3 h in total.

In [ ]:
# code: copy the newest echogesture dataset into /kaggle/working (idempotent)
import os, glob, shutil, subprocess
W = '/kaggle/working/echogesture'
src = sorted(glob.glob('/kaggle/input/**/echogesture/src/main.py', recursive=True))
assert src, 'Add your echogesture dataset as an input'
shutil.rmtree(W, ignore_errors=True)
shutil.copytree(os.path.dirname(os.path.dirname(src[0])), W)
os.chdir(W); print(os.getcwd())
print('code version:', open('src/VERSION').read().strip() if os.path.exists('src/VERSION') else 'OLD (no VERSION file)')

In [ ]:
!pip install -q mediapipe==1.2.2 ai-edge-litert==2.3.0 2>&1 | tail -2
!python src/tools/fetch_models.py

In [ ]:
# quick check on 40 clips BEFORE the long run: errors must be 0.
# The last progress line shows ms/frame and graph builds; builds should stay at a few per worker.
VID = glob.glob('/kaggle/input/**/wlasl-processed/videos', recursive=True)[0]; print(VID)
!rm -rf /tmp/lm_test
!python src/tools/extract_landmarks.py --videos {VID} --out /tmp/lm_test --subset 100 --trimmed --workers 4 --every 20 --limit 40
import numpy as np, glob as g
fs = g.glob('/tmp/lm_test/*.npy'); print(len(fs), 'test files')
if fs:
    a = np.load(fs[0]); print('example shape', a.shape, 'non-NaN fraction', float(np.isfinite(a.astype('float32')).mean()))

In [ ]:
# WLASL: all 2000 glosses (about 12k clips). Resumable: re-running skips finished clips.
!python src/tools/extract_landmarks.py --videos {VID} --out /kaggle/working/data/landmarks --subset 2000 --trimmed --workers 4

In [ ]:
# pack WLASL datasets for training
D = '/kaggle/working/data'
for k in (100, 300, 2000):
    !python src/tools/build_wlasl_dataset.py --landmarks {D}/landmarks --subset {k} --split official --out {D}/wlasl{k}.npz
!python src/tools/build_wlasl_dataset.py --landmarks {D}/landmarks --subset 300 --split signer --out {D}/wlasl300_si.npz
!python src/tools/build_wlasl_dataset.py --landmarks {D}/landmarks --subset 100 --split signer --out {D}/wlasl100_si.npz

In [ ]:
# Kaggle GISLR landmarks (250 signs, about 94k clips) for pretraining, about 30-40 min
ROOT = glob.glob('/kaggle/input/**/sign_to_prediction_index_map.json', recursive=True)[0].rsplit('/', 1)[0]; print(ROOT)
!python src/tools/gislr_to_packed.py --root {ROOT} --out {D}/gislr.npz

In [ ]:
# summary (copy this output into the chat)
!ls -la {D} | head -20
!ls {D}/landmarks | wc -l
!head -5 {D}/landmarks/_problems.csv; wc -l {D}/landmarks/_problems.csv
!du -sh {D}